# Traffic Classification & Regression Model Training
This notebook trains classification pipelines (for congestion level prediction) and regression pipelines (for traffic volume prediction) using Scikit-Learn and XGBoost.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from xgboost import XGBClassifier

train_df = pd.read_csv('../datasets/processed/train.csv')
test_df = pd.read_csv('../datasets/processed/test.csv')
print("Loaded train size:", len(train_df))
print("Loaded test size:", len(test_df))

## Classification Task: Predict Traffic Situation
We classify the traffic level as low, normal, high, or heavy.

In [ ]:
features = ['CarCount', 'BikeCount', 'BusCount', 'TruckCount', 'Hour', 'DayOfWeek', 'IsWeekend', 'IsPeakHour', 'DensityScore']
target = 'Traffic Situation'

mapping = {'low': 0, 'normal': 1, 'high': 2, 'heavy': 3}
X_train = train_df[features]
y_train = train_df[target].map(mapping)
X_test = test_df[features]
y_test = test_df[target].map(mapping)

# Random Forest Pipeline
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])
rf_pipeline.fit(X_train, y_train)

y_pred = rf_pipeline.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['low', 'normal', 'high', 'heavy']))

## XGBoost Classifier

In [ ]:
xgb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42))
])
xgb_pipeline.fit(X_train, y_train)
xgb_pred = xgb_pipeline.predict(X_test)
print("XGBoost Accuracy:", accuracy_score(y_test, xgb_pred))

## Feature Importance Analysis
Let's see which features contribute the most to the classification models.

In [ ]:
importances = xgb_pipeline.named_steps['model'].feature_importances_
feat_imp = pd.Series(importances, index=features).sort_values(ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='mako')
plt.title('XGBoost Classification Feature Importance')
plt.show()